# v0.2.8 live smoke test, run 2 (fixes 1): two Tata zones + one methodological validation zone

Runs the **v0.2.8 engine** on `EMU_Deccan_forest` (terrestrial) and **one valid Tata aquatic zone** (`Lake_Suman` unless the evidence step
shows it is unsuitable), then the parity harness. It does **not** run the 15 tiles, does **not** touch the headline aggregation and does
**not** touch `main`.

What changed since run 1 (all in branch `v0.2.8-contract`): every Earth Engine reference request is now bounded in size (tiled cell
references, tiled water-body vectorisation, adaptive split on a memory error); water-body area is a pixel count and edge bodies are
excluded by one shared rule; a failed water-body probe is `applicability_undetermined` (never "not applicable") and the evidence is
saved in the audit JSON (`meta.evidence`); the site value is the coverage-weighted polygon mean everywhere; the parity harness isolates
each check, reports masked bands instead of crashing, and adds a live **synthetic** water-body fixture painted inside Earth Engine.

Cell 9 is a **methodological validation** step on an existing forest-project tile (not a Tata zone, never in Tata scoring) because the
Deccan tile has ~0.01 ha of Hansen baseline forest, so forest-loss parity there is trivially true.

Send back: `*_audit.csv/json/md` for each zone, `parity.md/json`, and `evidence_and_tiling.txt` (the last cells produce and zip them).
Run the cells in order and keep the runtime alive between cells.

In [ ]:
# 1. Clone / update the v0.2.8 BRANCH (main is untouched). Safe to re-run.
import os, glob
REPO_DIR = '/content/reference-benchmarking'
BRANCH = 'v0.2.8-contract'
if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH https://github.com/G-auravSingh/reference-benchmarking.git $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin $BRANCH
    !git -C $REPO_DIR checkout $BRANCH
    !git -C $REPO_DIR pull --ff-only origin $BRANCH
PKG_DIR = sorted(glob.glob(f'{REPO_DIR}/darukaa_reference_v*'))[-1]
!git -C $REPO_DIR log --oneline -1
%cd $PKG_DIR
!pip install -q -r requirements.txt
!pip install -q -e .
!pip install -q pyproj shapely

In [ ]:
# 2. Earth Engine (same project as the main notebook)
import ee
ee.Authenticate()
ee.Initialize(project='gaurav-singh-007')
print('Earth Engine ready')

In [ ]:
# 3. Config: identical to the main notebook's Tata Motors config
from darukaa_reference.config import Config
from darukaa_reference.indicators import create_default_registry
config = Config(
    gee_project='gaurav-singh-007', output_dir='outputs', archetype='industrial',
    realm='terrestrial', assessment_mode='baseline',
    hmi_hard_ceiling=0.05, use_variance_stability_floor=True, reference_stratification='ecoregion_landcover',
)
registry = create_default_registry()
print('riparian ring width (m):', config.riparian_ring_width_m, '| early period:', config.net_change_early_years)

In [ ]:
# 4. Locate the two zones from the Tata manifests
import sys, os
sys.path.insert(0, os.getcwd())
from run_project_from_manifest import find_manifest_by_project_name, load_manifest, resolve_tile_paths
REPO_ROOT = os.path.dirname(os.getcwd())
PATHS = {}
for proj in ('TataMotors_Pimpri', 'TataMotors_Pimpri_Aquatic'):
    mp = find_manifest_by_project_name(REPO_ROOT, proj)
    man = load_manifest(mp)
    PATHS.update(dict(zip(man['tile_labels'], resolve_tile_paths(man, mp))))
TERRESTRIAL_LABEL, AQUATIC_LABEL = 'EMU_Deccan_forest', 'Lake_Suman'
print(TERRESTRIAL_LABEL, '->', PATHS[TERRESTRIAL_LABEL]); print(AQUATIC_LABEL, '->', PATHS[AQUATIC_LABEL])

## Step A. Aquatic zone selection (evidence only, cheap)
Measures the water body the aquatic indicators would use: water-mask area inside the site, the target body's pixel-count area, bounding
box and **pure-water pixel count**, and the ecosystem classification. `Lake_Suman` is used if it satisfies the open-water definition
(a complete water body with >= 10 pure-water pixels); otherwise the next existing Tata aquatic tile that does is used. A probe that
*fails* is shown as an error, never as "no water".

In [ ]:
# 5. Evidence for the aquatic candidates (evidence only: no reference building). Nothing here is invented: candidates are the existing Tata aquatic tiles.
import json, logging
logging.basicConfig(level=logging.WARNING)
from darukaa_reference import assess as A
CANDIDATES = ['Lake_Suman', 'Lake_Sharma', 'Pond_1', 'Pond_2', 'Pond_3', 'Pond_4']
CANDIDATES = [c for c in CANDIDATES if c in PATHS] or [k for k in PATHS if 'Lake' in k or 'Pond' in k]
selected, evidence_log = None, {}
for lab in CANDIDATES:
    z = A.load_zone(config, registry, PATHS[lab], lab, 'aquatic')
    prov = A.EEProvider(config, registry)
    ev = prov.evidence(z)
    evidence_log[lab] = {'water_probe': ev.water_probe, 'water_probe_detail': ev.water_probe_detail, 'water_body_valid': ev.water_body_valid,
                         'n_pure_water_px': ev.n_pure_water_px, 'invalid_reason': ev.water_body_invalid_reason,
                         'ecosystem_tags': sorted(ev.ecosystem_tags), 'site_area_m2': ev.site_area_m2, 'diagnostics': ev.diagnostics}
    d = ev.diagnostics
    print(f"\n=== {lab}: site {ev.site_area_m2/1e4:.2f} ha | probe={ev.water_probe} {ev.water_probe_detail} | valid={ev.water_body_valid} | pure px={ev.n_pure_water_px} | classification={d.get('ecosystem_classification')}")
    print('    water mask:', json.dumps(d.get('water_mask'), default=str)[:400])
    print('    body probe:', json.dumps(d.get('water_body_probe'), default=str)[:700])
    if d.get('errors'): print('    ERRORS:', d['errors'])
    if selected is None and ev.water_probe == 'ok' and ev.water_body_valid and (ev.n_pure_water_px or 0) >= 10:
        selected = lab
        break                                    # first zone that satisfies the definition: stop, do not touch more tiles
AQUATIC_LABEL = selected
print('\nSELECTED AQUATIC ZONE:', AQUATIC_LABEL)
assert AQUATIC_LABEL, 'no existing Tata aquatic tile satisfies the open-water definition: STOP and report the table above'

## Step B. Smoke test A: terrestrial zone

In [ ]:
# 6. SMOKE TEST A: EMU_Deccan_forest (writes outputs/smoke/EMU_Deccan_forest_audit.{csv,json,md}; the JSON has meta.evidence)
import time
TERRESTRIAL_LABEL = 'EMU_Deccan_forest'
zoneA = A.load_zone(config, registry, PATHS[TERRESTRIAL_LABEL], TERRESTRIAL_LABEL, 'terrestrial')
print('ecoregion id:', zoneA.eco_id)
providerA = A.EEProvider(config, registry)
rowsA = A.run_smoke_test(config, registry, zoneA, out_dir='outputs/smoke', provider=providerA)

## Step C. Smoke test B: the selected aquatic zone

In [ ]:
# 7. SMOKE TEST B: the aquatic zone selected in Step A
zoneB = A.load_zone(config, registry, PATHS[AQUATIC_LABEL], AQUATIC_LABEL, 'aquatic')
print('ecoregion id:', zoneB.eco_id)
providerB = A.EEProvider(config, registry)
rowsB = A.run_smoke_test(config, registry, zoneB, out_dir='outputs/smoke', provider=providerB)

## Step D. Methodological validation zone (forest loss)
Measures Hansen baseline forest and primary-window loss inside the **existing** Odisha forest-project tiles (`FCF_GV`, `FCF_Soova`) and
picks the one that gives the parity check real power (>= 5 ha baseline **and** non-zero loss). It is labelled a methodological validation
dataset and never enters Tata scoring. If none qualifies the cell says so and stops; nothing is invented.

In [ ]:
# 8. Choose and assess the forest validation zone (forest_loss_rate only)
cands = A.find_candidate_tiles(REPO_ROOT, top=6)
print('candidate tiles (existing assets):'); [print('  ', c['label'], round(c['area_ha'], 1), 'ha', c['project']) for c in cands]
zones_v = [A.load_zone(config, registry, c['path'], c['label'], 'terrestrial') for c in cands]
zoneV, vtable = A.select_forest_validation_zone(providerA, zones_v)
print(json.dumps(vtable, indent=1))
if zoneV is None:
    print('NO existing candidate has >= 5 ha baseline AND non-zero loss: forest-loss parity stays unvalidated. Report this table.')
else:
    print('SELECTED VALIDATION ZONE:', zoneV.label, '|', zoneV.validation_label)
    rowsV = A.run_smoke_test(config, registry, zoneV, out_dir='outputs/smoke', provider=providerA, only=['forest_loss_rate'])

## Step E. Parity
`DISCREPANCY` = a real finding; `HARNESS_ERROR` = the check itself failed (not evidence about the indicators); `INFO` = quantified, not judged.
Includes the **live synthetic water-body fixture** and, if a validation zone was found, the Hansen bundle and site-rate parity on it.

In [ ]:
# 9. EE-vs-numpy parity (each check isolated; masked bands reported, not fatal)
from darukaa_reference import parity
results = parity.run_parity(config, registry, zoneA, zoneB, out_dir='outputs/smoke', zone_forest=(zoneV if 'zoneV' in globals() else None))

In [ ]:
# 10. What the requests cost and how the tiling behaved (for the discrepancy report)
import json, glob
lines = []
for f in sorted(glob.glob('outputs/smoke/*_audit.json')):
    d = json.load(open(f)); m = d.get('meta', {}).get('evidence', {})
    lines.append(f"== {f}\nevidence: probe={m.get('water_probe')} {m.get('water_probe_detail')} valid={m.get('water_body_valid')} pure_px={m.get('n_pure_water_px')} tags={m.get('ecosystem_tags')}")
    lines.append(json.dumps(m.get('diagnostics', {}), indent=1, default=str)[:3000])
    for r in d['rows']:
        if r['seconds'] > 30 or r['reason'].startswith('provider_error') or r['reason'] == 'applicability_undetermined':
            prov = json.loads(r['provenance']) if isinstance(r['provenance'], str) else r['provenance']
            lines.append(f"  {r['indicator']:32s} {r['status']:32s} {r['reason'][:60]:60s} {r['seconds']:8.1f}s  {r.get('detail','')[:160]}")
open('outputs/smoke/evidence_and_tiling.txt', 'w').write('\n'.join(lines)); print('\n'.join(lines)[:6000])

In [ ]:
# 11. Zip everything to send back
import shutil
shutil.make_archive('/content/v028_smoke_results_run2', 'zip', 'outputs/smoke')
from google.colab import files
files.download('/content/v028_smoke_results_run2.zip')